# 05b: inference

Turns the test DICOMs into submission.csv. Runs with Internet off.

MODE = "model": average the fold checkpoints of every run in RUNS over every study.
RUNS lists the 05a experiments to ensemble. Each run must bring exactly 5 fold checkpoints,
so every run gets the same weight. Checkpoints from a run not in RUNS stop the notebook.
MODE = "prior": every study gets 0.5 (pipeline smoke test).
A study that cannot be decoded or predicted gets the per-finding median of the other test
predictions, and a 0.5 prior submission is written first, so a later failure still leaves
a valid file.

Inputs: competition data, 00-dicom-decoders output, one 05a output per run in RUNS (best_f*.pt),
utility scripts knee_prep and knee_model.
Settings: GPU T4 x2 (one is used), Internet off, Environment pinned to original.

Flip models: left knees are mirrored before prediction. The side comes from the DICOM tag
or scanner geometry, and for the rest from the 05c laterality model (lat_model.pt).
Attach the 05c output whose laterality model matches the models' cache geometry
(05c for px models, 05c-laterality-mm for mm models). The notebook stops if they do not match.

# Cell 1: imports, prior submission, decoders, models

In [ ]:
import os, sys, glob, time, subprocess
import numpy as np
import pandas as pd

ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
SPLIT = "test_series"
MODE = "model"
RUNS = ["b0-v1-mm-flip", "b2-v1-mm-flip"]
T0 = time.time()

def find_all(pattern):
    hits = glob.glob(f"/kaggle/usr/lib/**/{pattern}", recursive=True)
    for top in os.listdir("/kaggle/input"):
        if top != "competitions":
            hits += glob.glob(f"/kaggle/input/{top}/**/{pattern}", recursive=True)
    return sorted(hits)

for name in ["knee_prep", "knee_model"]:
    hits = find_all(f"{name}.py")
    print(name, hits)
    if not hits:
        raise FileNotFoundError(f"{name}.py not attached")
    sys.path.insert(0, os.path.dirname(hits[0]))
import knee_prep as kp
import knee_model as km

test = pd.read_csv(f"{ROOT}/test.csv")
sample = pd.read_csv(f"{ROOT}/sample_submission.csv")
PRIOR = np.full(len(km.LABELS), 0.5)
km.submission_frame(test["StudyInstanceUID"], [PRIOR] * len(test)).to_csv("/kaggle/working/submission.csv", index=False)
print("prior submission written for", len(test), "studies")

wheels = [d for d in find_all("wheels") if os.path.isdir(d)]
if wheels:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", wheels[0],
                        "pylibjpeg", "pylibjpeg-libjpeg", "pylibjpeg-openjpeg", "pylibjpeg-rle",
                        "python-gdcm"], capture_output=True, text=True)
    print("decoder install return code:", r.returncode, r.stderr[-300:])
else:
    print("decoder wheels not found")

MODELS, DEVICE = [], "cpu"
if MODE == "model":
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    by_run = {}
    for p in find_all("best_f*.pt"):
        exp = torch.load(p, map_location="cpu")["cfg"]["exp"]
        by_run.setdefault(exp, []).append(p)
    for exp, ps in sorted(by_run.items()):
        print(exp, len(ps), "checkpoints from", sorted({os.path.dirname(p) for p in ps}))
    extra = set(by_run) - set(RUNS)
    if extra:
        raise RuntimeError(f"checkpoints from runs not in RUNS are attached: {sorted(extra)}")
    for exp in RUNS:
        if len(by_run.get(exp, [])) != 5:
            raise RuntimeError(f"{exp}: {len(by_run.get(exp, []))} checkpoints instead of 5")
    try:
        MODELS = km.load_models([p for exp in RUNS for p in sorted(by_run[exp])], DEVICE)
    except Exception as err:
        print("model load failed:", repr(err)[:300])
    print(len(MODELS), "models on", DEVICE)

GEOMETRY = "px"
LAT_MODEL = None
if MODELS:
    geos = {m.geometry for m in MODELS}
    if len(geos) > 1:
        raise RuntimeError(f"models trained on different cache geometries: {geos}")
    GEOMETRY = geos.pop()
    if GEOMETRY != "px" and not hasattr(kp, "resample_mm"):
        raise RuntimeError("models need geometry mm but the attached knee_prep is too old: update it")
    print("geometry:", GEOMETRY, "| flip models:", sum(m.flip for m in MODELS), "of", len(MODELS))
    if any(m.flip for m in MODELS):
        lat_cks = []
        for p in find_all("lat_model.pt"):
            try:
                ck = torch.load(p, map_location="cpu")
                lat_cks.append((ck.get("geometry", "px"), p, ck))
            except Exception as err:
                print("cannot read", p, repr(err)[:200])
        print("laterality models found:", [(g, p) for g, p, _ in lat_cks])
        match = [x for x in lat_cks if x[0] == GEOMETRY]
        if not match:
            raise RuntimeError(f"flip models need a laterality model trained on {GEOMETRY} slices: "
                               f"attach the matching 05c output")
        lat_geo, lat_path, ck = match[0]
        LAT_MODEL = km.make_lat_net(pretrained=False)
        LAT_MODEL.load_state_dict(ck["state_dict"])
        LAT_MODEL = LAT_MODEL.to(DEVICE).eval()
        print("using laterality model:", lat_path, "| geometry", lat_geo)

# Cell 2: scan test series and laterality

In [ ]:
SIDE = {}
inv = None
try:
    ser = pd.read_csv(f"{ROOT}/test_series.csv")
    inv = kp.scan_all(ser, ROOT, SPLIT)
    lat = kp.study_laterality(inv)
    SIDE = lat.set_index("StudyInstanceUID")["side"].dropna().to_dict()
    print(f"{len(test)} studies, {len(ser)} series, scan errors {inv['error'].notna().sum()}, "
          f"elapsed {time.time() - T0:.0f}s")
    print(lat["side_src"].value_counts().to_string())
except Exception as err:
    print("scan failed:", repr(err)[:300])

# Cell 3: decode, resample and predict every study

In [ ]:
preds, log, OK = {}, [], set()
if inv is not None:
    try:
        kw = {"geometry": GEOMETRY} if GEOMETRY != "px" else {}
        for sid, vols, notes, sec in kp.iter_studies(inv, ROOT, SPLIT, workers=4, **kw):
            p, side, src = PRIOR, SIDE.get(sid), "dicom"
            if side not in ("L", "R"):
                side, src = None, "none"
                if LAT_MODEL is not None:
                    arr = km.lat_input(vols)
                    if arr is not None:
                        try:
                            pl = float(km.lat_predict(LAT_MODEL, arr[None], DEVICE)[0])
                            side, src = ("L" if pl > 0.5 else "R"), "classifier"
                        except Exception as err:
                            notes = notes + [f"laterality: {type(err).__name__}"]
            if MODELS and vols:
                try:
                    p = km.predict_study(MODELS, vols, DEVICE, side=side)
                    OK.add(sid)
                except Exception as err:
                    notes = notes + [f"predict: {type(err).__name__}"]
            preds[sid] = p
            log.append({"StudyInstanceUID": sid, "planes": "".join(sorted(k[0] for k in vols)),
                        "n_planes": len(vols), "side": side, "side_src": src,
                        "notes": "; ".join(notes), "sec": round(sec, 2)})
    except Exception as err:
        print("pipeline stopped:", repr(err)[:300])

log = pd.DataFrame(log, columns=["StudyInstanceUID", "planes", "n_planes", "side", "side_src", "notes", "sec"])
el = time.time() - T0
print("studies processed:", len(log), "of", len(test))
print(log["n_planes"].value_counts().to_string())
print("side source:", log["side_src"].value_counts().to_dict(), "| sides:", log["side"].value_counts(dropna=False).to_dict())
print("studies with fallback notes:", int((log["notes"] != "").sum()))
for n in log.loc[log["notes"] != "", "notes"].head(10):
    print("  ", n)
print(f"mean {log['sec'].mean():.2f}s per study per worker, total elapsed {el:.0f}s")

# Cell 4: write and check submission.csv

Studies that could not be predicted get the per-finding median of the predicted studies,
not 0.5: model outputs sit mostly well below 0.5, so 0.5 would rank a failed study as abnormal.

In [ ]:
FALLBACK = np.median([preds[s] for s in OK], axis=0) if OK else PRIOR
print("studies predicted:", len(OK), "of", len(test), "| fallback for the rest:", np.round(FALLBACK, 3).tolist())
probs = [preds[sid] if sid in OK else FALLBACK for sid in test["StudyInstanceUID"]]
sub = km.submission_frame(test["StudyInstanceUID"], probs)
problems = km.check_submission(sub, sample, test["StudyInstanceUID"])
print("problems:", problems or "none")
if not problems:
    sub.to_csv("/kaggle/working/submission.csv", index=False)
print(sub.round(3).head().to_string())
print(f"total {time.time() - T0:.0f}s")